In [ ]:
# Import libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

root = Path.cwd()
for _ in range(6):
    if (root / 'report').exists() or (root / 'data').exists():
        break
    root = root.parent

print(" Performing Financial Analytics...")

# 1. Market Research: Downloading financial data
print(" Downloading market data (Market Research)...")
ticker = "AAPL"
data = yf.download(ticker, start="2020-01-01", end="2024-01-31")

print("Downloaded data shape:", data.shape)
display(data.head())

# 2. Financial Analytics: Calculate key metrics
print(" Performing Financial Analytics...")
data['Daily Return'] = data['Close'].pct_change()
data['Volatility'] = data['Daily Return'].rolling(window=30).std()
data['MA_50'] = data['Close'].rolling(window=50).mean()
data['MA_200'] = data['Close'].rolling(window=200).mean()

# 3. Trend Analysis: Identify trends using moving averages
print(" Performing Trend Analysis...")
data['Trend'] = 'Neutral'
data.loc[data['MA_50'] > data['MA_200'], 'Trend'] = 'Upward'
data.loc[data['MA_50'] < data['MA_200'], 'Trend'] = 'Downward'

# Display trend analysis results
trend_counts = data['Trend'].value_counts()
print("Trend Analysis Results:")
print(trend_counts)

# Plot financial data
plt.figure(figsize=(14, 10))

plt.subplot(3, 1, 1)
plt.plot(data.index, data['Close'], label='Close Price', linewidth=1.5)
plt.plot(data.index, data['MA_50'], label='50-Day MA', alpha=0.7, linewidth=1.5)
plt.plot(data.index, data['MA_200'], label='200-Day MA', alpha=0.7, linewidth=1.5)
plt.title('AAPL Stock Price with Moving Averages', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylabel('Price ($)')

plt.subplot(3, 1, 2)
plt.plot(data.index, data['Volatility'], label='Volatility', color='red', linewidth=1)
plt.title('30-Day Volatility', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylabel('Volatility')

plt.subplot(3, 1, 3)
plt.bar(trend_counts.index, trend_counts.values, color=['green', 'gray', 'red'])
plt.title('Trend Distribution', fontsize=12, fontweight='bold')
plt.ylabel('Count')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 4. Save the analyzed data
financial_metrics = data[['Close', 'Daily Return', 'Volatility', 'MA_50', 'MA_200', 'Trend']].tail(100)

# Save to Excel
report_dir = root / 'report'
report_dir.mkdir(exist_ok=True)
with pd.ExcelWriter(report_dir / 'financial_analysis_report.xlsx', engine='openpyxl') as writer:
    financial_metrics.to_excel(writer, sheet_name='AAPL Analysis')
    pd.DataFrame(trend_counts).to_excel(writer, sheet_name='Trend Analysis')

print(" Financial Analytics Complete! Excel file saved to report/financial_analysis_report.xlsx.")
